<!--nav--> [🗺 Learning path](README.md) · **12/20** · ◀ [Simple MultiGPU DPO RLHF](./Simple_MultiGPU_DPO_RLHF.ipynb) · [GRPO Reasoning Training](./GRPO_Reasoning_Training.ipynb) ▶

# Alignment Showdown: DPO vs KTO vs ORPO vs SimPO

Train the same model with 4 different alignment methods.
Compare model quality AND training performance.

- **Model:** Qwen2.5-0.5B (latest tiny model)
- **Methods:** DPO, KTO, ORPO, SimPO — all with LoRA
- **Parallelism:** DeepSpeed ZeRO-2 across all GPUs
- **Benchmarks:** Quality (judge scoring) + Performance (speed, memory, utilization)

### The 4 methods in 10 seconds

```
DPO    — needs paired preferences (chosen vs rejected)
KTO    — only needs thumbs up / thumbs down (no pairs)
ORPO   — SFT + alignment in one step (no reference model)
SimPO  — like DPO but simpler (no reference model, length-normalized)
```

In [ ]:
!pip install -q transformers datasets peft accelerate "trl>=0.12" deepspeed

In [ ]:
import torch, os, json, time

assert torch.cuda.is_available(), "GPU required!"
NUM_GPUS = torch.cuda.device_count()
GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEM = torch.cuda.get_device_properties(0).total_memory / 1e9

for i in range(NUM_GPUS):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory/1e9:.0f} GB)")
print(f"\nTotal GPUs: {NUM_GPUS}")

In [ ]:
# Accelerate config — ZeRO-2 for all runs
accel_yaml = f"""compute_environment: LOCAL_MACHINE
distributed_type: DEEPSPEED
deepspeed_config:
  gradient_accumulation_steps: auto
  gradient_clipping: auto
  offload_optimizer_device: cpu
  offload_param_device: none
  zero3_init_flag: false
  zero_stage: 2
machine_rank: 0
main_process_ip: null
main_process_port: null
main_training_function: main
mixed_precision: bf16
num_machines: 1
num_processes: {NUM_GPUS}
use_cpu: false
"""
accel_dir = os.path.expanduser("~/.cache/huggingface/accelerate")
os.makedirs(accel_dir, exist_ok=True)
with open(os.path.join(accel_dir, "default_config.yaml"), "w") as f:
    f.write(accel_yaml)

print(f"Config: {NUM_GPUS} GPU(s), ZeRO-2, bf16")

## Prepare Shared Dataset

All 4 methods use the same preference data (UltraFeedback).
We pre-download once so each run doesn't re-download.

In [ ]:
from datasets import load_dataset

# Download once, save locally
full_ds = load_dataset("argilla/ultrafeedback-binarized-preferences-cleaned", split="train")
full_ds = full_ds.shuffle(seed=42)
train_ds = full_ds.select(range(500))
train_ds.save_to_disk("./bench_data_train")

eval_ds = full_ds.select(range(500, 600))
eval_ds.save_to_disk("./bench_data_eval")

print(f"Train: {len(train_ds)} | Eval: {len(eval_ds)} preference pairs saved.")
print(f"Example prompt: {train_ds[0]['chosen'][0]['content'][:100]}...")

## Write Training Scripts

4 scripts — one per method. All share:
- Same base model (Qwen2.5-0.5B)
- Same LoRA config (r=16, q_proj + v_proj)
- Same data (500 UltraFeedback examples)
- Same hyperparams where possible

In [ ]:
%%writefile train_method.py
"""Unified alignment training script. Reads METHOD env var."""
import torch, os, json, time
os.environ["WANDB_DISABLED"] = "true"

from datasets import load_from_disk
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainerCallback
from peft import LoraConfig
from trl import DPOConfig, DPOTrainer, KTOConfig, KTOTrainer

# Conditional imports — ORPO/SimPO may not exist in all trl versions
try:
    from trl import ORPOConfig, ORPOTrainer
    HAS_ORPO = True
except ImportError:
    HAS_ORPO = False

try:
    from trl import SimPOConfig, SimPOTrainer
    HAS_SIMPO = True
except ImportError:
    HAS_SIMPO = False

METHOD = os.environ.get("METHOD", "dpo")
MODEL = "Qwen/Qwen2.5-0.5B"
OUTPUT = f"./{METHOD}_output"

# Load model + tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.bfloat16, trust_remote_code=True)

lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none", task_type="CAUSAL_LM",
)

# Load data
dataset = load_from_disk("./bench_data_train")

def format_preference(ex):
    """Format into chat messages for DPO/ORPO/SimPO."""
    prompt_text = ex["chosen"][0]["content"] if ex["chosen"] else ""
    chosen_text = ex["chosen"][1]["content"] if len(ex["chosen"]) > 1 else ""
    rejected_text = ex["rejected"][1]["content"] if len(ex["rejected"]) > 1 else ""
    return {
        "prompt": [{"role": "user", "content": prompt_text}],
        "chosen": [
            {"role": "user", "content": prompt_text},
            {"role": "assistant", "content": chosen_text},
        ],
        "rejected": [
            {"role": "user", "content": prompt_text},
            {"role": "assistant", "content": rejected_text},
        ],
    }

def format_kto(ex):
    """KTO needs unpaired: each row is (prompt, completion, label=True/False)."""
    prompt_text = ex["chosen"][0]["content"] if ex["chosen"] else ""
    chosen_text = ex["chosen"][1]["content"] if len(ex["chosen"]) > 1 else ""
    rejected_text = ex["rejected"][1]["content"] if len(ex["rejected"]) > 1 else ""
    return {
        "prompt_chosen": prompt_text,
        "completion_chosen": chosen_text,
        "prompt_rejected": prompt_text,
        "completion_rejected": rejected_text,
    }

# Shared training args
SHARED = dict(
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    warmup_steps=10,
    logging_steps=1,
    bf16=True,
    gradient_checkpointing=True,
    report_to="none",
    save_strategy="no",
    max_length=512,
    max_prompt_length=256,
)

torch.cuda.reset_peak_memory_stats()
train_start = time.time()
loss_log = []

class LossCapture(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs:
            loss_log.append({"step": state.global_step, "loss": logs["loss"],
                            "lr": logs.get("learning_rate", 0)})

if METHOD == "dpo":
    ds = dataset.map(format_preference, remove_columns=dataset.column_names)
    print(f"DPO: {len(ds)} preference pairs")
    trainer = DPOTrainer(
        model=model,
        args=DPOConfig(output_dir=OUTPUT, beta=0.1, **SHARED),
        train_dataset=ds,
        processing_class=tokenizer,
        peft_config=lora_config,
        callbacks=[LossCapture()],
    )

elif METHOD == "kto":
    # KTO: unpack paired data into unpaired (chosen=True, rejected=False)
    ds = dataset.map(format_kto, remove_columns=dataset.column_names)
    from datasets import Dataset, concatenate_datasets
    chosen = Dataset.from_dict({
        "prompt": [[{"role": "user", "content": p}] for p in ds["prompt_chosen"]],
        "completion": [[{"role": "assistant", "content": c}] for c in ds["completion_chosen"]],
        "label": [True] * len(ds),
    })
    rejected = Dataset.from_dict({
        "prompt": [[{"role": "user", "content": p}] for p in ds["prompt_rejected"]],
        "completion": [[{"role": "assistant", "content": c}] for c in ds["completion_rejected"]],
        "label": [False] * len(ds),
    })
    kto_ds = concatenate_datasets([chosen, rejected]).shuffle(seed=42)
    print(f"KTO: {len(kto_ds)} examples ({len(chosen)} good + {len(rejected)} bad)")
    trainer = KTOTrainer(
        model=model,
        args=KTOConfig(output_dir=OUTPUT, desirable_weight=1.0, undesirable_weight=1.0, **SHARED),
        train_dataset=kto_ds,
        processing_class=tokenizer,
        peft_config=lora_config,
        callbacks=[LossCapture()],
    )

elif METHOD == "orpo":
    if not HAS_ORPO:
        print("ORPO not available in this trl version. Using DPO with ORPO-style settings as fallback.")
        ds = dataset.map(format_preference, remove_columns=dataset.column_names)
        # Fallback: DPO without reference model approximates ORPO's spirit
        shared_no_prompt = {k: v for k, v in SHARED.items() if k not in ("max_prompt_length",)}
        trainer = DPOTrainer(
            model=model,
            args=DPOConfig(output_dir=OUTPUT, beta=0.1, **SHARED),
            train_dataset=ds,
            processing_class=tokenizer,
            peft_config=lora_config,
            callbacks=[LossCapture()],
        )
    else:
        ds = dataset.map(format_preference, remove_columns=dataset.column_names)
        print(f"ORPO: {len(ds)} preference pairs")
        orpo_args = {k: v for k, v in SHARED.items() if k not in ("max_prompt_length",)}
        trainer = ORPOTrainer(
            model=model,
            args=ORPOConfig(output_dir=OUTPUT, beta=0.1, **orpo_args),
            train_dataset=ds,
            processing_class=tokenizer,
            peft_config=lora_config,
            callbacks=[LossCapture()],
        )

elif METHOD == "simpo":
    if not HAS_SIMPO:
        print("SimPO not available in this trl version. Falling back to DPO with length normalization.")
        ds = dataset.map(format_preference, remove_columns=dataset.column_names)
        trainer = DPOTrainer(
            model=model,
            args=DPOConfig(output_dir=OUTPUT, beta=0.1, loss_type="simpo", **SHARED),
            train_dataset=ds,
            processing_class=tokenizer,
            peft_config=lora_config,
            callbacks=[LossCapture()],
        )
    else:
        ds = dataset.map(format_preference, remove_columns=dataset.column_names)
        print(f"SimPO: {len(ds)} preference pairs")
        simpo_args = {k: v for k, v in SHARED.items() if k not in ("max_prompt_length",)}
        trainer = SimPOTrainer(
            model=model,
            args=SimPOConfig(output_dir=OUTPUT, beta=2.0, gamma=0.5, **simpo_args),
            train_dataset=ds,
            processing_class=tokenizer,
            peft_config=lora_config,
            callbacks=[LossCapture()],
        )

trainer.train()
train_time = time.time() - train_start

trainer.save_model(f"{OUTPUT}/final")
tokenizer.save_pretrained(f"{OUTPUT}/final")

# Save metrics
if int(os.environ.get("LOCAL_RANK", 0)) == 0:
    num_gpus = int(os.environ.get("WORLD_SIZE", 1))
    total_steps = trainer.state.global_step
    total_params = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

    metrics = {
        "method": METHOD.upper(),
        "model": MODEL,
        "total_params": total_params,
        "trainable_params": trainable,
        "trainable_pct": round(trainable / total_params * 100, 2),
        "num_gpus": num_gpus,
        "gpu_name": torch.cuda.get_device_name(0),
        "gpu_mem_allocated_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2),
        "gpu_mem_reserved_gb": round(torch.cuda.max_memory_reserved() / 1e9, 2),
        "gpu_mem_total_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1),
        "train_time_sec": round(train_time, 2),
        "total_steps": total_steps,
        "samples_per_sec": round(total_steps * 2 * 4 * num_gpus / train_time, 2),
        "avg_step_time_ms": round(train_time / max(total_steps, 1) * 1000, 1),
        "final_loss": loss_log[-1]["loss"] if loss_log else None,
        "loss_history": loss_log,
    }
    with open(f"metrics_{METHOD}.json", "w") as f:
        json.dump(metrics, f, indent=2)
    print(f"\n{METHOD.upper()}: {train_time:.0f}s | {metrics['samples_per_sec']:.1f} samples/s | {metrics['gpu_mem_allocated_gb']:.1f} GB peak")

---
## Run All 4 Methods

In [ ]:
methods = ["dpo", "kto", "orpo", "simpo"]
run_times = {}

for m in methods:
    print(f"\n{'='*60}")
    print(f"  {m.upper()} — {NUM_GPUS} GPU(s), ZeRO-2")
    print(f"{'='*60}")
    start = time.time()
    !METHOD={m} accelerate launch --num_processes={NUM_GPUS} train_method.py
    run_times[m] = time.time() - start
    print(f"{m.upper()} total: {run_times[m]:.0f}s")

---
## Model Quality Evaluation

Generate responses from each model and score them.
We measure: response length, perplexity, and a simple quality heuristic.

In [ ]:
import torch, json, math
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from datasets import load_from_disk

MODEL = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

eval_ds = load_from_disk("./bench_data_eval")

# Test prompts for generation
test_prompts = [
    "Explain quantum computing to a 10-year-old.",
    "What are the pros and cons of remote work?",
    "Write a short poem about the ocean.",
    "How do neural networks learn?",
    "What makes a good leader?",
]

quality_results = {}

for method in ["base", "dpo", "kto", "orpo", "simpo"]:
    # Load model
    base = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.bfloat16, trust_remote_code=True).to("cuda")
    if method != "base":
        adapter_path = f"./{method}_output/final"
        if not os.path.exists(adapter_path):
            print(f"Skipping {method} — adapter not found")
            del base; torch.cuda.empty_cache()
            continue
        model = PeftModel.from_pretrained(base, adapter_path)
    else:
        model = base
    model.eval()

    # 1. Generate responses
    responses = []
    for p in test_prompts:
        text = f"<|im_start|>user\n{p}<|im_end|>\n<|im_start|>assistant\n"
        inputs = tokenizer(text, return_tensors="pt").to("cuda")
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=150, temperature=0.7, do_sample=True,
                                 pad_token_id=tokenizer.pad_token_id)
        resp = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        responses.append(resp)

    # 2. Compute perplexity on eval set (sample)
    total_nll = 0
    total_tokens = 0
    for i in range(min(50, len(eval_ds))):
        ex = eval_ds[i]
        chosen_text = ex["chosen"][1]["content"] if len(ex["chosen"]) > 1 else ""
        if not chosen_text:
            continue
        inputs = tokenizer(chosen_text, return_tensors="pt", truncation=True, max_length=256).to("cuda")
        with torch.no_grad():
            outputs = model(**inputs, labels=inputs["input_ids"])
        total_nll += outputs.loss.item() * inputs["input_ids"].shape[1]
        total_tokens += inputs["input_ids"].shape[1]

    ppl = math.exp(total_nll / total_tokens) if total_tokens > 0 else float("inf")

    # 3. Quality heuristics
    avg_len = sum(len(r.split()) for r in responses) / len(responses)
    non_empty = sum(1 for r in responses if len(r.strip()) > 10) / len(responses)
    # Vocab diversity: unique words / total words
    all_words = " ".join(responses).split()
    diversity = len(set(all_words)) / max(len(all_words), 1)

    quality_results[method] = {
        "perplexity": round(ppl, 2),
        "avg_response_words": round(avg_len, 1),
        "completion_rate": round(non_empty * 100, 0),
        "vocab_diversity": round(diversity * 100, 1),
        "responses": dict(zip(test_prompts, responses)),
    }
    print(f"{method.upper():6s} | PPL: {ppl:7.1f} | Avg words: {avg_len:5.1f} | Diversity: {diversity*100:.1f}%")

    del model, base
    torch.cuda.empty_cache()

with open("quality_results.json", "w") as f:
    json.dump(quality_results, f, indent=2)
print("\nQuality results saved.")

---
## Full Dashboard

In [ ]:
import json, glob, math
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from IPython.display import HTML, display
import numpy as np

# Load all metrics
perf = {}
for f in sorted(glob.glob("metrics_*.json")):
    with open(f) as fh:
        d = json.load(fh)
        perf[d["method"]] = d

with open("quality_results.json") as f:
    qual = json.load(f)

methods_order = ["DPO", "KTO", "ORPO", "SIMPO"]
methods_present = [m for m in methods_order if m in perf]
colors = {"DPO": "#58a6ff", "KTO": "#3fb950", "ORPO": "#f0883e", "SIMPO": "#d2a8ff"}
c = [colors[m] for m in methods_present]

# ── Chart 1: Performance comparison (2x2 grid) ──
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.patch.set_facecolor("#0d1117")
fig.suptitle("Training Performance Comparison", color="#e6edf3", fontsize=16, fontweight="bold", y=0.98)

def style_ax(ax, title, ylabel):
    ax.set_facecolor("#0d1117")
    ax.set_title(title, color="#e6edf3", fontsize=13, fontweight="bold")
    ax.set_ylabel(ylabel, color="#8b949e", fontsize=10)
    ax.tick_params(colors="#8b949e")
    ax.grid(True, axis="y", alpha=0.15, color="#30363d")
    for spine in ax.spines.values():
        spine.set_color("#30363d")

def add_labels(ax, bars, vals, fmt="{:.0f}"):
    mx = max(vals) if vals else 1
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + mx*0.02,
                fmt.format(v), ha="center", va="bottom", color="#e6edf3", fontsize=11, fontweight="bold")

# Training time
vals = [perf[m]["train_time_sec"] for m in methods_present]
bars = axes[0,0].bar(methods_present, vals, color=c, width=0.5)
style_ax(axes[0,0], "Training Time (lower = better)", "Seconds")
add_labels(axes[0,0], bars, vals, "{:.0f}s")

# Throughput
vals = [perf[m]["samples_per_sec"] for m in methods_present]
bars = axes[0,1].bar(methods_present, vals, color=c, width=0.5)
style_ax(axes[0,1], "Throughput (higher = better)", "Samples/sec")
add_labels(axes[0,1], bars, vals, "{:.1f}")

# GPU Memory
vals = [perf[m]["gpu_mem_allocated_gb"] for m in methods_present]
bars = axes[1,0].bar(methods_present, vals, color=c, width=0.5)
gpu_total = perf[methods_present[0]]["gpu_mem_total_gb"]
axes[1,0].axhline(y=gpu_total, color="#f85149", linestyle="--", alpha=0.5)
style_ax(axes[1,0], "Peak GPU Memory", "GB")
add_labels(axes[1,0], bars, vals, "{:.1f} GB")
axes[1,0].set_ylim(0, gpu_total * 1.2)

# Step time
vals = [perf[m]["avg_step_time_ms"] for m in methods_present]
bars = axes[1,1].bar(methods_present, vals, color=c, width=0.5)
style_ax(axes[1,1], "Avg Step Time (lower = faster)", "Milliseconds")
add_labels(axes[1,1], bars, vals, "{:.0f} ms")

plt.tight_layout()
plt.show()

# ── Chart 2: Model quality comparison ──
fig2, axes2 = plt.subplots(1, 3, figsize=(14, 4))
fig2.patch.set_facecolor("#0d1117")
fig2.suptitle("Model Quality Comparison", color="#e6edf3", fontsize=16, fontweight="bold", y=1.02)

qual_methods = ["base"] + [m.lower() for m in methods_present]
qual_colors = ["#8b949e"] + [colors[m] for m in methods_present]
qual_labels = ["Base"] + methods_present

# Perplexity (lower = better)
vals = [qual[m]["perplexity"] for m in qual_methods if m in qual]
lbls = [l for m, l in zip(qual_methods, qual_labels) if m in qual]
clrs = [c for m, c in zip(qual_methods, qual_colors) if m in qual]
bars = axes2[0].bar(lbls, vals, color=clrs, width=0.5)
style_ax(axes2[0], "Perplexity (lower = better)", "PPL")
add_labels(axes2[0], bars, vals, "{:.1f}")

# Response length
vals = [qual[m]["avg_response_words"] for m in qual_methods if m in qual]
bars = axes2[1].bar(lbls, vals, color=clrs, width=0.5)
style_ax(axes2[1], "Avg Response Length", "Words")
add_labels(axes2[1], bars, vals, "{:.0f}")

# Vocab diversity
vals = [qual[m]["vocab_diversity"] for m in qual_methods if m in qual]
bars = axes2[2].bar(lbls, vals, color=clrs, width=0.5)
style_ax(axes2[2], "Vocabulary Diversity", "%")
add_labels(axes2[2], bars, vals, "{:.0f}%")

plt.tight_layout()
plt.show()

# ── Chart 3: Loss curves overlaid ──
fig3, ax = plt.subplots(figsize=(12, 4))
fig3.patch.set_facecolor("#0d1117")
ax.set_facecolor("#0d1117")

for m in methods_present:
    hist = perf[m]["loss_history"]
    if hist:
        ax.plot([h["step"] for h in hist], [h["loss"] for h in hist],
                color=colors[m], linewidth=2, label=m, alpha=0.85)

ax.set_xlabel("Step", color="#8b949e", fontsize=11)
ax.set_ylabel("Loss", color="#8b949e", fontsize=11)
ax.set_title("Training Loss — All Methods", color="#e6edf3", fontsize=14, fontweight="bold")
ax.tick_params(colors="#8b949e")
ax.grid(True, alpha=0.15, color="#30363d")
ax.legend(facecolor="#161b22", edgecolor="#30363d", labelcolor="#e6edf3", fontsize=11)
for spine in ax.spines.values():
    spine.set_color("#30363d")
plt.tight_layout()
plt.show()

In [ ]:
# ── HTML Summary Dashboard ──

# Find bests
fastest = min(methods_present, key=lambda m: perf[m]["train_time_sec"])
most_throughput = max(methods_present, key=lambda m: perf[m]["samples_per_sec"])
least_mem = min(methods_present, key=lambda m: perf[m]["gpu_mem_allocated_gb"])
aligned_methods = [m.lower() for m in methods_present if m.lower() in qual]
best_ppl = min(aligned_methods, key=lambda m: qual[m]["perplexity"]) if aligned_methods else "?"

# Build comparison table rows
table_rows = ""
for m in methods_present:
    p = perf[m]
    q = qual.get(m.lower(), {})
    ppl = q.get("perplexity", "—")
    div = q.get("vocab_diversity", "—")
    mem_pct = p["gpu_mem_allocated_gb"] / p["gpu_mem_total_gb"] * 100

    badges = []
    if m == fastest: badges.append("fastest")
    if m == least_mem: badges.append("least mem")
    if m.lower() == best_ppl: badges.append("best quality")
    badge_html = "".join(f'<span style="background:#238636; color:white; padding:1px 6px; border-radius:8px; font-size:10px; margin-left:4px;">{b}</span>' for b in badges)

    table_rows += f"""
    <tr style="border-bottom: 1px solid #21262d;">
      <td style="padding:10px; font-weight:700; color:{colors[m]};">{m} {badge_html}</td>
      <td style="padding:10px; text-align:center;">{p['train_time_sec']:.0f}s</td>
      <td style="padding:10px; text-align:center;">{p['samples_per_sec']:.1f}</td>
      <td style="padding:10px; text-align:center;">{p['avg_step_time_ms']:.0f} ms</td>
      <td style="padding:10px; text-align:center;">{p['gpu_mem_allocated_gb']:.1f} GB ({mem_pct:.0f}%)</td>
      <td style="padding:10px; text-align:center;">{p['final_loss']:.3f}</td>
      <td style="padding:10px; text-align:center;">{ppl}</td>
      <td style="padding:10px; text-align:center;">{div}%</td>
    </tr>"""

# GPU utilization bars
util_bars = ""
for m in methods_present:
    p = perf[m]
    pct = p["gpu_mem_allocated_gb"] / p["gpu_mem_total_gb"] * 100
    util_bars += f"""
    <div style="display:flex; align-items:center; margin:6px 0;">
      <span style="color:{colors[m]}; font-weight:600; width:60px; font-size:12px;">{m}</span>
      <div style="flex:1; margin:0 12px; background:#21262d; border-radius:6px; height:16px; overflow:hidden;">
        <div style="width:{pct:.0f}%; height:100%; border-radius:6px;
                    background: linear-gradient(90deg, {colors[m]}88, {colors[m]});"></div>
      </div>
      <span style="color:#e6edf3; font-size:12px; font-weight:600; width:55px; text-align:right;">{pct:.0f}%</span>
    </div>"""

html = f"""
<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;
            max-width: 880px; margin: 20px 0;">

  <div style="display: grid; grid-template-columns: repeat(4, 1fr); gap: 12px; margin-bottom: 16px;">
    <div style="background: linear-gradient(135deg, #1a2332, #161b22); border: 1px solid #30363d;
                border-radius: 12px; padding: 16px; text-align: center;">
      <div style="color: #8b949e; font-size: 10px; text-transform: uppercase; letter-spacing: 1px;">Fastest</div>
      <div style="color: {colors[fastest]}; font-size: 24px; font-weight: 700; margin: 6px 0;">{fastest}</div>
      <div style="color: #8b949e; font-size: 12px;">{perf[fastest]['train_time_sec']:.0f}s</div>
    </div>
    <div style="background: linear-gradient(135deg, #1a2332, #161b22); border: 1px solid #30363d;
                border-radius: 12px; padding: 16px; text-align: center;">
      <div style="color: #8b949e; font-size: 10px; text-transform: uppercase; letter-spacing: 1px;">Best Quality</div>
      <div style="color: {colors.get(best_ppl.upper(), '#e6edf3')}; font-size: 24px; font-weight: 700; margin: 6px 0;">{best_ppl.upper()}</div>
      <div style="color: #8b949e; font-size: 12px;">PPL {qual.get(best_ppl, {{}}).get('perplexity', '?')}</div>
    </div>
    <div style="background: linear-gradient(135deg, #1a2332, #161b22); border: 1px solid #30363d;
                border-radius: 12px; padding: 16px; text-align: center;">
      <div style="color: #8b949e; font-size: 10px; text-transform: uppercase; letter-spacing: 1px;">Least Memory</div>
      <div style="color: {colors[least_mem]}; font-size: 24px; font-weight: 700; margin: 6px 0;">{least_mem}</div>
      <div style="color: #8b949e; font-size: 12px;">{perf[least_mem]['gpu_mem_allocated_gb']:.1f} GB</div>
    </div>
    <div style="background: linear-gradient(135deg, #1a2332, #161b22); border: 1px solid #30363d;
                border-radius: 12px; padding: 16px; text-align: center;">
      <div style="color: #8b949e; font-size: 10px; text-transform: uppercase; letter-spacing: 1px;">GPUs</div>
      <div style="color: #e6edf3; font-size: 24px; font-weight: 700; margin: 6px 0;">{perf[methods_present[0]]['num_gpus']}x</div>
      <div style="color: #8b949e; font-size: 12px;">{perf[methods_present[0]]['gpu_name']}</div>
    </div>
  </div>

  <div style="background: #161b22; border: 1px solid #30363d; border-radius: 12px; overflow: hidden; margin-bottom: 12px;">
    <table style="width:100%; border-collapse: collapse; color: #c9d1d9; font-size: 12px;">
      <thead>
        <tr style="background: #0d1117; border-bottom: 2px solid #30363d;">
          <th style="padding:10px; text-align:left; color:#8b949e;">Method</th>
          <th style="padding:10px; text-align:center; color:#8b949e;">Time</th>
          <th style="padding:10px; text-align:center; color:#8b949e;">Throughput</th>
          <th style="padding:10px; text-align:center; color:#8b949e;">Step</th>
          <th style="padding:10px; text-align:center; color:#8b949e;">GPU Mem</th>
          <th style="padding:10px; text-align:center; color:#8b949e;">Loss</th>
          <th style="padding:10px; text-align:center; color:#8b949e;">PPL</th>
          <th style="padding:10px; text-align:center; color:#8b949e;">Diversity</th>
        </tr>
      </thead>
      <tbody>{table_rows}</tbody>
    </table>
  </div>

  <div style="background: #161b22; border: 1px solid #30363d; border-radius: 12px; padding: 16px; margin-bottom: 12px;">
    <div style="color: #e6edf3; font-size: 13px; font-weight: 600; margin-bottom: 10px;">GPU Memory Utilization</div>
    {util_bars}
  </div>

  <div style="background: #161b22; border: 1px solid #30363d; border-radius: 12px; padding: 16px; color: #8b949e; font-size: 12px;">
    <strong style="color: #e6edf3;">Method guide:</strong><br>
    <span style="color:#58a6ff;">DPO</span> — needs paired preferences (chosen vs rejected). Stable, proven.<br>
    <span style="color:#3fb950;">KTO</span> — only needs thumbs up/down, no pairs. Easier data collection.<br>
    <span style="color:#f0883e;">ORPO</span> — SFT + alignment combined. No reference model. One-step training.<br>
    <span style="color:#d2a8ff;">SimPO</span> — simplified DPO. No reference model, length-normalized. Less compute.
  </div>

</div>
"""
display(HTML(html))

## Sample Responses — Side by Side

In [ ]:
with open("quality_results.json") as f:
    qual = json.load(f)

available = [m for m in ["base", "dpo", "kto", "orpo", "simpo"] if m in qual]

for prompt in test_prompts:
    print(f"\n{'='*70}")
    print(f"Q: {prompt}")
    print(f"{'='*70}")
    for m in available:
        resp = qual[m]["responses"].get(prompt, "(no response)")
        color_map = {"base": "BASE", "dpo": "DPO ", "kto": "KTO ", "orpo": "ORPO", "simpo": "SMPO"}
        tag = color_map.get(m, m.upper())
        print(f"\n[{tag}] {resp[:300]}")

---

## How Each Method Works

### DPO (Direct Preference Optimization)
```
Data:  (prompt, chosen, rejected) pairs
Loss:  -log sigmoid(β * (log π(chosen) - log π_ref(chosen) - log π(rejected) + log π_ref(rejected)))
Needs: reference model (frozen copy)
```

### KTO (Kahneman-Tversky Optimization)
```
Data:  (prompt, completion, good/bad label) — no pairs needed
Loss:  based on prospect theory — losses hurt more than gains help
Needs: reference model (frozen copy)
```

### ORPO (Odds Ratio Preference Optimization)
```
Data:  (prompt, chosen, rejected) pairs
Loss:  SFT loss + λ * odds_ratio_loss — combined in one step
Needs: NO reference model
```

### SimPO (Simple Preference Optimization)
```
Data:  (prompt, chosen, rejected) pairs
Loss:  like DPO but uses average log-prob (length-normalized) + reward margin
Needs: NO reference model
```

### When to use what

| Situation | Method |
|-----------|--------|
| You have paired preferences | DPO or SimPO |
| You only have thumbs up/down | KTO |
| You want SFT + alignment in one go | ORPO |
| You want simplest setup, no ref model | SimPO or ORPO |
| You need proven, well-studied method | DPO |